# Experiment 5: Perceptron vs Multilayer Perceptron with Hyperparameter Tuning

Name: Kushaal Shyam Potta

Reg. No: 3122235001071


In [1]:
import pandas as pd
import numpy as np
import os
import cv2
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

1. Preprocess the dataset (resize, flatten, normalize).

In [ ]:
img_folder = r"dataset/Img"
df = pd.read_csv("dataset/english.csv")

# List to store dimensions
dimensions = []

for path in df['image']:
    # Get the filename and full path
    img_file = os.path.basename(path)
    img_path = os.path.join(img_folder, img_file)
    
    # Load image in grayscale as specified in the experiment
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    
    if img is not None:
        h, w = img.shape  # This gives you (height, width)
        dimensions.append((h, w))
    else:
        dimensions.append((None, None))


In [ ]:
print(dimensions)

#all images are of dimension 900x1200

[(900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200), (900, 1200)

In [9]:
def load_and_process_data(csv_path, img_folder, target_size=(40,30)):
    df = pd.read_csv(csv_path)
    X,y=[],[]

    print("Downscaling and Flattening images...")

    for index,row in df.iterrows():
        img_path=os.path.join(img_folder, os.path.basename(row['image']))
        img=cv2.imread(img_path,cv2.IMREAD_GRAYSCALE)

        if img is not None:
            img_resized=cv2.resize(img, target_size)
            img_flattened = img_resized.flatten()
            X.append(img_flattened/255.0)
            y.append(row['label'])

    return np.array(X), np.array(y)

X,y=load_and_process_data("dataset/english.csv", img_folder)

le=LabelEncoder()
y_encoded=le.fit_transform(y)
num_classes=len(le.classes_)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.3, random_state=42, shuffle=True, stratify=y_encoded)


Downscaling and Flattening images...


In [8]:
print(num_classes)

62


2. Implement PLA from scratch:

• Use step activation function.

• Apply perceptron weight update rule.

• Extend PLA to multi-class classification using One-vs-Rest

In [ ]:
class PLA:
    def __init__(self,learning_rate=0.01,epochs=10):
        self.lr=learning_rate
        self.epochs = epochs
        self.weights=None
        self.bias=0

    def step_activation(self, z):
        return 1 if z>=0 else 0
    
    def fit(self, X,y):
        num_samples, num_features= X.shape
        self.weights=np.zeroes(num_features)
        self.bias=0

        for epoch in range(self.epochs):
            for i,x_i in enumerate(X):
                linear_output = np.dot(x_i,self.weights) + self.bias

                y_predicted=self.step_activation(linear_output)

                update = self.lr*(y[i]-y_predicted)
                self.weights+=update*x_i
                self.bias+=update

    def get_score(self,X):
        """Return raw dot product to determine confidence for OvR"""
        return np.dot(X,self.weights) + self.bias



In [ ]:
def step_activation(self, z):
        """Standard step function: returns 1 if z >= 0, else 0."""
        return 1 if z >= 0 else 0

    def fit(self, X, y):
        """
        Implements the PLA weight update rule: 
        w = w + lr * (y - y_pred) * x
        """
        n_samples, n_features = X.shape
        # Initialize weights to zeros
        self.weights = np.zeros(n_features)
        self.bias = 0

        for epoch in range(self.epochs):
            for idx, x_i in enumerate(X):
                # Calculate linear output (z)
                linear_output = np.dot(x_i, self.weights) + self.bias
                # Apply step activation
                y_predicted = self.step_activation(linear_output)
                
                # Weight update rule 
                update = self.lr * (y[idx] - y_predicted)
                self.weights += update * x_i
                self.bias += update

    def get_score(self, X):
        """Returns the raw dot product to determine confidence for OvR."""
        return np.dot(X, self.weights) + self.bias